---
# 0. Setup

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder, TargetEncoder
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# import pyspark as spark

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:,.2f}'.format)
sns.set_theme(style='whitegrid')

# Notebooks live in EDA/, data lives in <repo root>/data/ (git-ignored -- never commit it).
DATA_DIR = Path('..') / 'data'

PAX_PATH = DATA_DIR / 'AA_PAX_FORECASTING.csv'

AIRPRT_PATH = DATA_DIR / 'AA_AIRPORT_INFO.csv'

---
# 1.  Load



In [ ]:
pax_raw = pd.read_csv(
    PAX_PATH,
    index_col=0,   # nameless exported row-number column -> index, not a feature
    dtype={'ORG_AIRPRT': 'category', 'DES_AIRPRT': 'category'},
)
pax = pax_raw.copy()
print(pax.shape)
print(f"{pax.memory_usage(deep=True).sum() / 1e6:,.0f} MB")
pax.drop(columns=['SF_LOAD_TMS'], inplace=True)
pax.head()

airprt_raw = pd.read_csv(
    AIRPRT_PATH,
    index_col=0,
    dtype={'ORG_AIRPRT': 'category', 'DES_AIRPRT': 'category'},
)
airprt = airprt_raw.copy()
print(airprt.shape)
print(f"{airprt.memory_usage(deep=True).sum() / 1e6:,.0f} MB")
airprt.drop(columns=['SF_LOAD_TMS'], inplace=True)


---
# 2. Previews


In [ ]:
pax.head()

In [ ]:
airprt.head()

In [ ]:
print(airprt.info())
print(airprt.describe(include='all'))
print(airprt.nunique())

In [ ]:
print(pax.info())
print(pax.describe(include='all'))
print(pax.nunique())


## Pax
Very heavily skewed OD_PAX, OD_REV_USD. The rest are rather evenly distributed (airports). The month and year are just dates split into two. 

## Airprt
Contains a lot of metrics on an airport, including their location, runways length, elevation, name, etc. Most of it is just informational. 

---
# 3 Imputing Data

### 3.1 Missing values, duplicates, and the key

In [ ]:

print(pax.duplicated().sum())
print(pax[pax['ORG_AIRPRT'] == pax['DES_AIRPRT']])
print(pax.isnull().value_counts())


In [ ]:
# print(airprt.duplicated().sum())
airprt.isnull().value_counts()
print(airprt.CNTRY_CD.isnull().value_counts())
print(airprt.STATE_PROVNC_CD.isnull().value_counts())


Let's fill in the NaN's with MISSING.

In [ ]:
airprt.STATE_PROVNC_CD = airprt.STATE_PROVNC_CD.fillna("MISSING").replace("","MISSING")
airprt.CNTRY_CD = airprt.CNTRY_CD.fillna("MISSING").replace("","MISSING")
airprt.isnull().value_counts()


### 3.2 Explore -- Pax

In [ ]:

for col in pax.columns:
    
    print(col,pax[col].value_counts().sort_values(ascending=False))

In [ ]:

key = ["DEP_YEAR","DEP_MONTH"]
pax_num = pax.select_dtypes(include="number")
num_cols = pax_num.columns.drop(key, errors="ignore")
cat_cols = [c for c in pax.columns if c not in pax_num.columns and c != key]

out = pax.groupby( key).agg({
    **{c: "mean" for c in num_cols},
    **{c: "count" for c in cat_cols},   # first value in each group
})
# print(out)

print(out.describe())
print(pax_num.head())


Usually few thousands of passengers per month (Normally ~2000-3000+) with hundreds of thousands (normally 500k - 850k+) of revenue dollars. 

In [ ]:

pax['DATE'] = pd.to_datetime(dict(year=pax['DEP_YEAR'], month=pax['DEP_MONTH'], day=1))
out = pax.groupby("DATE").mean(numeric_only=True).sort_index()
out.reset_index(inplace=True)
print(out)
pax.groupby('DATE').size().plot()



In [ ]:
#Let's also add the average revenue
pax['AVG_FARE'] = pax['OD_REV_USD'] / pax['OD_PAX']
print(pax['AVG_FARE'].describe())
pax['AVG_FARE'].plot(kind='hist')

In [ ]:
#Logging the average fare because it's so skewed
pax['AVG_FARE_LOG'] = np.log1p(pax.AVG_FARE)
pax.AVG_FARE_LOG.plot(kind='hist',bins =30)
print(pax.AVG_FARE_LOG.describe())


The log looks a lot better, so let's do that to the PAX and REV_USD too. 

In [ ]:
pax['OD_REV_USD_LOG'] = np.log1p(pax.OD_REV_USD)
pax['OD_PAX_LOG'] = np.log1p(pax.OD_PAX)
print(pax.OD_REV_USD_LOG.describe())
print(pax.OD_PAX_LOG.describe())

In [ ]:
plt.figure(figsize = (10,6))
pax.OD_PAX.plot(kind='hist',alpha=0.5, color='blue',bins=30)
pax.OD_REV_USD.plot(kind='hist',alpha=0.5, color='red',bins=30)

In [ ]:

pax.OD_PAX_LOG.plot(kind='hist',alpha=0.5, color='blue',bins=30,label='PAX')
pax.OD_REV_USD_LOG.plot(kind='hist',alpha=0.5, color='red',bins=30,label='REVENUE')
plt.legend()
plt.show()
# pax.OD_PAX.plot(kind='hist',alpha=0.5, color='blue',bins=30,label='PAX')
# pax.OD_REV_USD.plot(kind='hist',alpha=0.5, color='red',bins=30,label='REVENUE')
# plt.xscale('log')
# plt.legend()
# plt.show()


In [ ]:
pax.plot(kind='box',column = ['OD_PAX','OD_REV_USD','AVG_FARE'])

In [ ]:
pax.plot(kind='box',column = ['OD_PAX_LOG','OD_REV_USD_LOG','AVG_FARE_LOG'])

A lot more normal. We'll use these for calculations.

In [ ]:
plt.figure(figsize=(10,6))
out.plot(x='DATE',y='OD_PAX', kind='line')
plt.xlabel("DATE")
plt.ylabel("PASSENGERS")
plt.title("Passengers Over Time")
plt.show()
plt.figure(figsize=(10,6))
out.plot(x='DATE',y='OD_REV_USD',kind='line')
plt.xlabel("DATE")
plt.ylabel("REVENUE (USD)")
plt.title("Revenue Over Time")
plt.show()

In [ ]:
# TODO: panel completeness. For each route (ORG, DES), how many of the possible months does it have?
#   route_months = pax.groupby(['ORG_AIRPRT', 'DES_AIRPRT'], observed=True)['DATE'].nunique()
#   route_months.describe(); route_months.hist(bins=50)
# Routes with gaps matter: a missing month could mean "0 passengers" or "no data". Which is it here?
# (observed=True stops pandas building every possible category pair -- 150 x 150 combos.)

pax['ROUTE'] = pax['ORG_AIRPRT'].astype(str) + '-' + pax['DES_AIRPRT'].astype(str)
print(pax.ROUTE.value_counts())
print(pax.nunique())
# print(pax.ROUTE.nunique())



In [ ]:
pax.groupby('ROUTE')[['OD_PAX','OD_REV_USD','OD_PAX_LOG','OD_REV_USD_LOG']].mean().sort_values(by=['OD_PAX','OD_REV_USD'],ascending=False)



In [ ]:
date = pax.pop('DATE')
route = pax.pop('ROUTE')
pax.insert(2,'DATE',date)
pax.insert(5,'ROUTE',route)
# pax.sort_values(by='DATE',inplace=True)
# pax.sort_index(inplace=True)
pax.head()

## 3.3 Explore and Modify -- Airprt

In [ ]:
airprt.head()

In [ ]:
def dms_to_decimal(df, deg_col, min_col, sec_col, code_col):
    """
    Converts columns of Degrees, Minutes, Seconds, and Direction Code 
    into a single series of plain Decimal Degrees.
    """
    # 1. Calculate absolute decimal degree value: Deg + (Min / 60) + (Sec / 3600)
    decimal_degrees = df[deg_col] + (df[min_col] / 60.0) + (df[sec_col] / 3600.0)
    
    # 2. Make coordinates negative if they are South or West
    # .str.upper().str.strip() ensures whitespace or casing variations don't break the logic
    direction_modifier = df[code_col].str.upper().str.strip().map({'N': 1, 'E': 1, 'S': -1, 'W': -1})
    
    # Fill any unexpected codes with 1 so it doesn't break, then multiply
    return decimal_degrees * direction_modifier.fillna(1)

def process_coordinates(df, lat_cols=None, lon_cols=None):
    """
    Takes a dataframe and maps DMS columns into plain 'latitude' and 'longitude' columns.
    
    Parameters:
      lat_cols: list/tuple of 4 strings for Lat -> (Deg, Min, Sec, Code)
      lon_cols: list/tuple of 4 strings for Lon -> (Deg, Min, Sec, Code)
    """
    # Default column name assumptions if none are provided
    lat_cols = lat_cols or ['lat_deg', 'lat_min', 'lat_sec', 'lat_code']
    lon_cols = lon_cols or ['lon_deg', 'lon_min', 'lon_sec', 'lon_code']
    
    df_copy = df.copy()
    
    # Apply transformation for Latitude and Longitude
    df_copy['LAT'] = dms_to_decimal(df_copy, *lat_cols).round(6)
    df_copy['LNGTD'] = dms_to_decimal(df_copy, *lon_cols).round(6)
    
    return df_copy

In [ ]:
pre_airprt = airprt.copy()
airprt = process_coordinates(airprt, lat_cols=['LAT_DEGREE_QTY','LAT_MINUTE_QTY','LAT_SECOND_QTY','LAT_HEMSPHR_CD'],
                             lon_cols=['LNGTD_DEGREE_QTY','LNGTD_MINUTE_QTY','LNGTD_SECOND_QTY','LNGTD_HEMSPHR_CD'])

In [ ]:
pd.set_option('display.float_format', lambda x: f'{x:.6f}')
airprt.drop(columns=['LAT_DEGREE_QTY','LAT_MINUTE_QTY','LAT_SECOND_QTY','LAT_HEMSPHR_CD',
                     'LNGTD_DEGREE_QTY','LNGTD_MINUTE_QTY','LNGTD_SECOND_QTY','LNGTD_HEMSPHR_CD'], inplace=True)
airprt.head()


In [ ]:
print(airprt[airprt['AIRPRT_CD'] != airprt['CITY_METRO_IATA_CD']].value_counts())

Reduced feature count by merging degrees, minutes, seconds to just degrees. 

---
# 5 Joining

Joining pax and airprt on AIRPRT_CD with ORG_AIRPRT and DES_AIRPRT

In [ ]:
df_step1 = pax.merge(airprt.add_prefix('ORG_'),left_on='ORG_AIRPRT',right_on='ORG_AIRPRT_CD',how='left')
# df_step1.head()
df = df_step1.merge(airprt.add_prefix('DES_'),left_on='DES_AIRPRT',right_on='DES_AIRPRT_CD',how='left')


In [ ]:
df.head(15)

---
# 6 Visalize Again



In [ ]:

monthly = df.groupby('DATE')[['OD_PAX', 'OD_REV_USD']].mean()
monthly = monthly.resample('ME').mean()



plt.plot(monthly.index, monthly['OD_PAX'], label='OD_PAX')
# plt.axvline(x='2020-03-01', color='red', linestyle='--', label='COVID Shock')
# plt.yscale('log')
plt.xlabel('Date')
plt.ylabel('OD_PAX')
plt.title('Monthly OD_PAX Over Time')
plt.legend()
plt.show()
plt.plot(monthly.index, monthly['OD_REV_USD'], label='OD_REV_USD')
# plt.yscale('log')
plt.xlabel('Date')
plt.ylabel('OD_REV_USD')
plt.title('Monthly OD_REV_USD Over Time')
plt.legend()
plt.show()

In [ ]:

seasonality = df.pivot_table(
    index='DEP_MONTH',
    columns='DEP_YEAR',
    values='OD_PAX',
    aggfunc='mean',
    fill_value=0,
)

plt.plot(seasonality.index, seasonality, marker='o',label=seasonality.columns)
plt.xlabel('Departure Month')
plt.ylabel('Total OD_PAX')
plt.title('Monthly OD_PAX by Departure Year')
plt.xticks(seasonality.index)
plt.legend()
plt.show()

seasonality = df.pivot_table(
    index='DEP_MONTH',
    columns='DEP_YEAR',
    values='OD_REV_USD',
    aggfunc='mean',
    fill_value=0,
)

plt.plot(seasonality.index, seasonality, marker='o',label=seasonality.columns)
plt.xlabel('Departure Month')
plt.ylabel('Total OD_REV_USD')
plt.title('Monthly OD_REV_USD by Departure Year')
plt.xticks(seasonality.index)
plt.legend()
plt.show()


In [ ]:

seasonality = df.pivot_table(
    index='DEP_MONTH',
    columns='DEP_YEAR',
    values='AVG_FARE',
    aggfunc='mean',
    fill_value=0,
)

plt.plot(seasonality.index, seasonality, marker='o',label=seasonality.columns)
plt.xlabel('Departure Month')
plt.ylabel('Average Fare')
plt.title('Monthly Average Fare by Departure Year')
plt.xticks(seasonality.index)
plt.legend()
plt.show()
print(df.groupby('DATE')['AVG_FARE'].mean().describe())
print("GROUPED BY MONTH")
print(df.groupby("DEP_MONTH")["AVG_FARE"].mean())

I expected to find price and passenger spikes during summer break, but there were none/the difference was very unnoticeable.

---
# 7 Correlations

In [ ]:
df.head()

In [ ]:
df_copy = df.copy()
# df.drop(columns=['DATE','OD_PAX','OD_REV_USD','AVG_FARE','ORG_AIRPRT_NM','ORG_CITY_METRO_IATA_CD','ORG_WRLD_AREA_DOT_CD',
#                  'ORG_AIRPRT_CD','DES_AIRPRT_CD','DES_AIRPRT_NM','DES_CITY_METRO_IATA_CD','DES_WRLD_AREA_DOT_CD',
                #  'AVG_FARE_LOG'],inplace=True)


df.head()

In [ ]:
#preparing to encode categorical variables
y = df[['OD_PAX_LOG','OD_REV_USD_LOG']]
X = df.drop(columns=['OD_PAX_LOG','OD_REV_USD_LOG'])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:

numeric_cols = [
    'DEP_MONTH', 'DEP_YEAR', 'OD_REV_USD_LOG',
    'OD_PAX_LOG', 'ORG_LNGST_RUNWAY_FT_QTY', 'ORG_ELEVATN_FT_QTY', 'ORG_LAT',
    'ORG_LNGTD', 'DES_LNGST_RUNWAY_FT_QTY', 'DES_ELEVATN_FT_QTY', 'DES_LAT',
    'DES_LNGTD'
]

plt.figure(figsize=(10,6))
sns.heatmap(df[numeric_cols].corr(), annot=True, cmap='coolwarm', fmt=".2f")
plt.title("Numerical Feature Correlation Check")
plt.show()

---
# 8. Build Model

In [ ]:
import time

In [ ]:


def build_and_evaluate_flight_model(X_train, X_test, y_train, y_test, target_name):
    """
    Builds, cross-validates, and trains a pipeline for a log-transformed target.
    Automatically applies inverse-log transformation for real-world metrics.
    """
    print(f"\n=====================================================================")
    print(f"🚀 INITIALIZING LOG-AWARE TRAINING PIPELINE FOR: {target_name.upper()}")
    print(f"=====================================================================")
    
    # 1. SETUP MASTER REPRODUCIBLE STRING DICTIONARIES
    master_states = sorted(list(
        set(X_train['ORG_STATE_PROVNC_CD']).union(set(X_train['DES_STATE_PROVNC_CD']))
        .union(set(X_test['ORG_STATE_PROVNC_CD'])).union(set(X_test['DES_STATE_PROVNC_CD']))
    ))
    master_countries = sorted(list(
        set(X_train['ORG_CNTRY_CD']).union(set(X_train['DES_CNTRY_CD']))
        .union(set(X_test['ORG_CNTRY_CD'])).union(set(X_test['DES_CNTRY_CD']))
    ))
    master_airports = sorted(list(
        set(X_train['ORG_AIRPRT']).union(set(X_train['DES_AIRPRT']))
        .union(set(X_test['ORG_AIRPRT'])).union(set(X_test['DES_AIRPRT']))
    ))

    # 2. INITIALIZE ISOLATED OBJECT INSTANCES
    fresh_route_encoder = TargetEncoder(smooth='auto', cv=5, random_state=42)
    state_encoder = OrdinalEncoder(categories=[master_states, master_states], handle_unknown='use_encoded_value', unknown_value=-1)
    country_encoder = OrdinalEncoder(categories=[master_countries, master_countries], handle_unknown='use_encoded_value', unknown_value=-1)
    airport_encoder = OrdinalEncoder(categories=[master_airports, master_airports], handle_unknown='use_encoded_value', unknown_value=-1)
    
    preprocessor = ColumnTransformer(
        transformers=[
            ('route_target', fresh_route_encoder, ['ROUTE']),
            ('states_ordinal', state_encoder, ['ORG_STATE_PROVNC_CD', 'DES_STATE_PROVNC_CD']),
            ('countries_ordinal', country_encoder, ['ORG_CNTRY_CD', 'DES_CNTRY_CD']),
            ('airports_ordinal', airport_encoder, ['ORG_AIRPRT', 'DES_AIRPRT'])
        ],
        remainder='passthrough'
    )
    
    fresh_rf = RandomForestRegressor(
        n_estimators=30, max_depth=12, min_samples_split=50, n_jobs=8, random_state=42
    )

    pipeline = Pipeline(steps=[
        ('preprocessing', preprocessor),
        ('model', fresh_rf)
    ])

    # 3. RUN K-FOLD CROSS-VALIDATION (Evaluates log-scale baseline)
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    print(f"⏳ Running 5-Fold Cross-Validation on training pool...")
    cv_start = time.time()
    cv_scores = cross_val_score(pipeline, X_train, y_train[target_name], cv=kf, scoring='r2', n_jobs=1)
    cv_end = time.time()
    print(f"📊 K-Fold Log-Scale R² Scores: {np.round(cv_scores, 4)}")
    print(f"🏆 Mean K-Fold Log-Scale R²: {np.mean(cv_scores):.4f} (Took {(cv_end - cv_start) / 60:.2f} mins)")

    # 4. FINAL FIT 
    print(f"🎓 Training final model on entire training matrix...")
    pipeline.fit(X_train, y_train[target_name])
    
    # 5. INVERSE-LOG TEST EVALUATION (REAL WORLD UNITS)
    # Generate predictions (which will come out as logged numbers)
    y_pred_log = pipeline.predict(X_test)
    
    # Reverse the log transform to get back to dollars/minutes
    # (Using expm1 assuming targets were logged using log1p)
    y_true_real = np.expm1(y_test[target_name])
    y_pred_real = np.expm1(y_pred_log)
    
    # Calculate true, honest metrics on the real-world scale!
    real_world_mae = mean_absolute_error(y_true_real, y_pred_real)
    real_world_r2 = r2_score(y_true_real, y_pred_real)
    
    print(f"\n🎯 --- FINAL REAL-WORLD HOLDOUT METRICS ---")
    print(f"💵 Real-World Mean Absolute Error (MAE): {real_world_mae:.2f} (in original units)")
    print(f"📈 Real-World Final R² Score: {real_world_r2:.4f}")
    
    return pipeline


In [ ]:
model_predictions = {}
model_predictions['OD_REV_USD_PRED'] = build_and_evaluate_flight_model(X_train, X_test, y_train, y_test, 'OD_REV_USD_LOG')
model_predictions['OD_PAX_PRED'] = build_and_evaluate_flight_model(X_train, X_test, y_train, y_test, 'OD_PAX_LOG')


In [ ]:
# pred = pd.DataFrame(model_predictions)
# pred.head()
model_predictions

---
# 9 Summary 



### Methods
1. Loaded the dataset and inspected
2. Filled in missing values and dropped useless columns
3. Created a function that first:
4. Builds a pipeline for encoding categorical variables
5. Evaluates using KFold Validation
6. Train again on train set
7. Tests on test set and evaluates
